# Tick Data

Adapted from [ib_async tick_data](https://ib-api-reloaded.github.io/ib_async/notebooks.html).
Demonstrates streaming L1 market data and tick-by-tick data.

**Best run during market hours.**

In [ ]:
import os, threading, time
from dotenv import load_dotenv
from ibx import (
    EWrapper, EClient, Contract, TickTypeEnum,
    TickAttrib, TickAttribLast, TickAttribBidAsk,
)

load_dotenv()
USERNAME = os.environ["IB_USERNAME"]
PASSWORD = os.environ["IB_PASSWORD"]

In [ ]:
class App(EWrapper):
    def __init__(self):
        super().__init__()
        self.ticks = {}       # req_id -> {tick_type: value}
        self.tbt_trades = []  # tick-by-tick last trades
        self.tbt_quotes = []  # tick-by-tick bid/ask
        self.connected = threading.Event()

    def next_valid_id(self, order_id):
        self.connected.set()

    def tick_price(self, req_id, tick_type, price, attrib):
        self.ticks.setdefault(req_id, {})[tick_type] = price

    def tick_size(self, req_id, tick_type, size):
        self.ticks.setdefault(req_id, {})[tick_type] = size

    def tick_by_tick_all_last(self, req_id, tick_type, time_, price, size,
                              tick_attrib_last, exchange, special_conditions):
        self.tbt_trades.append((time_, price, size, exchange))

    def tick_by_tick_bid_ask(self, req_id, time_, bid_price, ask_price,
                             bid_size, ask_size, tick_attrib_bid_ask):
        self.tbt_quotes.append((time_, bid_price, ask_price, bid_size, ask_size))

    def error(self, req_id, error_code, error_string, advanced_order_reject_json=""):
        if error_code not in (2104, 2106, 2158):
            print(f"Error {error_code}: {error_string}")

In [ ]:
app = App()
client = EClient(app)
client.connect(username=USERNAME, password=PASSWORD, paper=True)

thread = threading.Thread(target=client.run, daemon=True)
thread.start()
app.connected.wait(timeout=10)
print(f"Connected: {client.is_connected()}")

### Streaming L1 Ticks

Subscribe to AAPL market data and collect ticks for 5 seconds:

In [ ]:
aapl = Contract(con_id=265598, symbol="AAPL")
client.req_mkt_data(1, aapl)

time.sleep(5)

TICK_NAMES = {
    TickTypeEnum.BID: "Bid", TickTypeEnum.ASK: "Ask", TickTypeEnum.LAST: "Last",
    TickTypeEnum.BID_SIZE: "BidSize", TickTypeEnum.ASK_SIZE: "AskSize",
    TickTypeEnum.LAST_SIZE: "LastSize", TickTypeEnum.VOLUME: "Volume",
    TickTypeEnum.HIGH: "High", TickTypeEnum.LOW: "Low",
    TickTypeEnum.OPEN: "Open", TickTypeEnum.CLOSE: "Close",
}

ticks = app.ticks.get(1, {})
for tick_type, value in sorted(ticks.items()):
    name = TICK_NAMES.get(tick_type, f"Type{tick_type}")
    print(f"  {name}: {value}")

### Live Updates (10 second loop)

Print a live-updating quote table:

In [ ]:
from IPython.display import clear_output

for _ in range(10):
    t = app.ticks.get(1, {})
    bid = t.get(TickTypeEnum.BID, 0)
    ask = t.get(TickTypeEnum.ASK, 0)
    last = t.get(TickTypeEnum.LAST, 0)
    vol = t.get(TickTypeEnum.VOLUME, 0)
    clear_output(wait=True)
    print(f"AAPL  Bid: {bid:.2f}  Ask: {ask:.2f}  Last: {last:.2f}  Vol: {vol:.0f}")
    time.sleep(1)

Cancel L1 subscription:

In [ ]:
client.cancel_mkt_data(1)

### Tick-by-Tick Data

Request raw tick-by-tick last trades for 5 seconds:

In [ ]:
app.tbt_trades.clear()
client.req_tick_by_tick_data(2, aapl, "Last")

time.sleep(5)
client.cancel_tick_by_tick_data(2)

print(f"Got {len(app.tbt_trades)} trades")
for ts, price, size, exchange in app.tbt_trades[:10]:
    print(f"  {price:.2f} x {size:.0f}  ({exchange})")

### Tick-by-Tick BidAsk

Request raw bid/ask ticks for 5 seconds:

In [ ]:
app.tbt_quotes.clear()
client.req_tick_by_tick_data(3, aapl, "BidAsk")

time.sleep(5)
client.cancel_tick_by_tick_data(3)

print(f"Got {len(app.tbt_quotes)} quotes")
for ts, bid, ask, bsz, asz in app.tbt_quotes[:10]:
    print(f"  {bid:.2f} x {bsz:.0f}  |  {ask:.2f} x {asz:.0f}")

In [ ]:
client.disconnect()